In [37]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import statsmodels.api as sms
from ipywidgets import interact, IntSlider, FloatSlider, RadioButtons, Dropdown, Checkbox, fixed, Layout, HBox, VBox, Accordion, interactive_output
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_squared_error, r2_score

In [38]:
# ----------------------------------------------------------------------
# 1. Define kernel functions
# ----------------------------------------------------------------------
def kernel_gaussian(u):
    return (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * u**2)

def kernel_epanechnikov(u):
    return np.where(np.abs(u) <= 1, 0.75 * (1 - u**2), 0)

def kernel_uniform(u):
    return np.where(np.abs(u) <= 1, 0.5, 0)

# Dictionary to select kernel by name
kernels = {
    'Gaussian': kernel_gaussian,
    'Epanechnikov': kernel_epanechnikov,
    'Uniform': kernel_uniform
}

In [39]:
# ----------------------------------------------------------------------
# 2. Generate synthetic data
# ----------------------------------------------------------------------
 
data1 = yf.download("^GSPC", start = "2015-01-01", end = "2025-12-31", interval = "1d")
prices1 = data1['Close'].values
data1_1 = prices1[1:len(prices1)] - prices1[0:(len(prices1)-1)]

data2 = yf.download("HDFCBANK.NS", start = "2015-01-01", end = "2025-12-31", interval = "1d")
prices2 = data2['Close'].values
data2_1 = prices2[1:len(prices2)] - prices2[0:(len(prices2)-1)]

data3 = yf.download("^NSEI", start = "2015-01-01", end = "2025-12-31", interval = "1d")
prices3 = data3['Close'].values
data3_1 = prices3[1:len(prices3)] - prices3[0:(len(prices3)-1)]

data4 = sms.datasets.get_rdataset("mcycle", "MASS").data
data4_1 = data4['accel'].values

# data5 = sms.datasets.get_rdataset('galaxies', 'MASS').data.values
data6 = pd.read_csv("C:\\Users\\Sumit\\Desktop\\Sumit\\python\\GLM_Assignment\\lidar.csv")
data6_1 = data6['logratio'].values

# --- synthetic dataset with outliers: normal + 3 far points ---
np.random.seed(42)
synthetic_outliers = np.concatenate([
    np.random.normal(0, 1, 200),
    np.array([5, 6, 7])   # clear outliers
])

datasets = {"S&P 500 Returns" : data1_1, "HDFC_BANK" : data2_1, "NIFTY_50" : data3_1, "Mcycle_Acceleration" : data4_1, "Laser_Velocity" : data6_1, "Synthetic (with outliers)" : synthetic_outliers}

hist_bins = list(range(1, 50, 2))

# n_kde = len(kde_data)

# ---- Data for regression: sin + noise, plus a linear part ----
n_reg = 150
x_reg = np.random.uniform(-3, 4, n_reg)
x_reg.sort()
true_m = np.sin(x_reg) + 0.5 * np.maximum(0, x_reg - 1)   # nonlinear function
y_reg = true_m + np.random.normal(0, 0.2, n_reg)

# Grid for evaluation
x_grid = np.linspace(-5, 7, 500)          # for KDE
x_grid_reg = np.linspace(-3, 4, 300)      # for regression

# True function on the fine grid (for plotting)
true_m_grid = np.sin(x_grid_reg) + 0.5 * np.maximum(0, x_grid_reg - 1)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [40]:
# ----------------------------------------------------------------------
# 3. KDE function (manual, to show kernel summations)
# ----------------------------------------------------------------------
def kde_estimate(x_grid, data, bandwidth, kernel_func, show_individual=False):
    """Return KDE estimates and optionally individual kernel contributions."""
    n = len(data)
    kde = np.zeros_like(x_grid)
    ind_kernels = []
    for i, xi in enumerate(x_grid):
        u = (xi - data) / bandwidth                         # Defining the variable u = (xi - X) / h
        w = kernel_func(u) / bandwidth                      # Defining the individual kernel/bandwidth: K(u)/h
        kde[i] = np.mean(w)                                 # To get the final KDM for the data -> i.e.  to get sum(K(u))/nh -> we'll directly take the mean of the list of K(u)/h -> mean(w)
        if show_individual:
            ind_kernels.append(w)
    if show_individual:
        return kde, np.array(ind_kernels).T  # shape (n_data, n_grid)
    return kde

In [41]:
# ----------------------------------------------------------------------
# 4. Nadaraya‑Watson regression function
# ----------------------------------------------------------------------
def nadaraya_watson(x_grid, x_data, y_data, bandwidth, kernel_func):
    y_est = np.zeros_like(x_grid)
    for i, x0 in enumerate(x_grid):
        u = (x0 - x_data) / bandwidth
        weights = kernel_func(u)
        if np.sum(weights) == 0:
            y_est[i] = np.nan
        else:
            y_est[i] = np.sum(weights * y_data) / np.sum(weights)
    return y_est

In [42]:
# ----------------------------------------------------------------------
# 5. Interactive plots (with Outlier Detection)
# ----------------------------------------------------------------------

def kde_at_points(points, data, bandwidth, kernel_func):
    """Compute KDE at specified points (e.g., the data points themselves)."""
    n = len(data)
    densities = np.zeros_like(points)
    for i, x0 in enumerate(points):
        u = (x0 - data) / bandwidth
        w = kernel_func(u) / bandwidth
        densities[i] = np.mean(w)
    return densities

def update_kde(bandwidth, kernel_name, show_individual, dataset_name, hist_b, show_outliers, outlier_percentile):
    kernel_func = kernels[kernel_name]
    kde_data = datasets[dataset_name]
    if show_individual:
        kde, ind_kernels = kde_estimate(x_grid, kde_data, bandwidth, kernel_func,
                                        show_individual=True)
    else:
        kde = kde_estimate(x_grid, kde_data, bandwidth, kernel_func,
                           show_individual=False)
    plt.figure(figsize=(10, 5))
    # Plot histogram of data (normalized to density)
    plt.hist(kde_data, bins=hist_b, density=True, alpha=0.4, color='gray',
             edgecolor='black', label='Data histogram')
    # Plot estimated density
    plt.plot(x_grid, kde, 'r-', linewidth=2, label=f'KDE (h={bandwidth:.2f})')

    # If requested, plot individual kernels (thin lines)
    if show_individual and len(ind_kernels) > 0:
        for i in range(len(kde_data)):
            plt.plot(x_grid, ind_kernels[i] / len(kde_data), 'b-', alpha=0.15, linewidth=0.5)

    # Optional: plot individual kernels
    if show_individual and len(ind_kernels) > 0:
        for i in range(len(kde_data)):
            plt.plot(x_grid, ind_kernels[i] / len(kde_data), 'b-', alpha=0.15, linewidth=0.5)

    # Outlier detection (if requested)
    if show_outliers:
        # Compute density at each data point
        densities_at_data = kde_at_points(kde_data, kde_data, bandwidth, kernel_func)

        # Define outlier threshold (e.g., 5th percentile of densities)
        threshold = np.percentile(densities_at_data, outlier_percentile * 100)
        outliers = kde_data[densities_at_data < threshold]

        # Mark outliers on the plot (bottom)
        # Get current y limits to place markers at a small negative offset
        ymin, ymax = plt.ylim()
        offset = -0.05 * (ymax - ymin)   # 5% below the lower limit
        plt.scatter(outliers, [offset] * len(outliers), color='red', s=300,
                    marker='^', edgecolor='darkred', zorder=5, label='Outliers')
        # Adjust y limit to show the markers
        plt.ylim(bottom=offset)

    plt.title(f'Kernel Density Estimation\nKernel = {kernel_name}, Bandwidth = {bandwidth:.2f}')
    plt.xlabel('x')
    plt.ylabel('Density')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.xlim(-5, 7)
    plt.show()



def update_reg(bandwidth, kernel_name):
    kernel_func = kernels[kernel_name]
    y_est = nadaraya_watson(x_grid_reg, x_reg, y_reg, bandwidth, kernel_func)

    plt.figure(figsize=(10, 5))
    plt.scatter(x_reg, y_reg, alpha=0.5, s=20, label='Data', color='gray')
    plt.plot(x_grid_reg, true_m_grid, 'g--', linewidth=2, label='True regression function')
    plt.plot(x_grid_reg, y_est, 'r-', linewidth=2, label=f'Nadaraya‑Watson (h={bandwidth:.2f})')
    plt.title(f'Kernel Regression (Nadaraya‑Watson)\nKernel = {kernel_name}, Bandwidth = {bandwidth:.2f}')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.xlim(-3, 4)
    plt.ylim(-1.5, 2.5)
    plt.show()


In [43]:
# ----------------------------------------------------------------------
# 6. Create interactive widgets
# ----------------------------------------------------------------------
# print("### Kernel Density Estimation (KDE) ###")
# interact(update_kde,
#         bandwidth=FloatSlider(min=0.1, max=1.5, step=0.05, value=0.4,
#                                description='Bandwidth h:'),
#         hist_b = IntSlider(min = 1, max = 50, step = 2, value = 30, description = 'Histogram Binwidth:'),
#         kernel_name=Dropdown(options=list(kernels.keys()), value='Gaussian',
#                               description='Kernel:'),
#         show_individual=Checkbox(value=False, description='Show individual kernels'),
#         dataset_name = RadioButtons(options = list(datasets.keys()), value = 'S&P 500 Returns',description = 'Dataset:'),
#         show_outliers=Checkbox(value=False, description='Show outliers'),
#         outlier_percentile=FloatSlider(min=0.01, max=0.2, step=0.01, value=0.05, description='Outlier percentile:'));



# Create widgets
bandwidth_slider = FloatSlider(min=0.1, max=1.5, step=0.05, value=0.4,
                               description='Bandwidth h:', layout=Layout(width='300px'))
hist_b_slider = IntSlider(min=1, max=50, step=2, value=30,
                          description='Histogram bins:', layout=Layout(width='250px'))
kernel_dropdown = Dropdown(options=list(kernels.keys()), value='Gaussian',
                           description='Kernel:')
show_individual_check = Checkbox(value=False, description='Show individual kernels')
dataset_radio = RadioButtons(options=list(datasets.keys()), value='S&P 500 Returns',
                             description='Dataset:', layout=Layout(width='200px'))

# Outlier controls (collapsible)
show_outliers_check = Checkbox(value=False, description='Show outliers')
outlier_percentile_slider = FloatSlider(min=0.01, max=0.2, step=0.01, value=0.05,
                                        description='Outlier percentile:', layout=Layout(width='300px'))
outlier_controls = VBox([show_outliers_check, outlier_percentile_slider])
outlier_accordion = Accordion(children=[outlier_controls])
outlier_accordion.set_title(0, 'Outlier detection')

# Arrange top row: bandwidth and histogram bins
top_row = HBox([bandwidth_slider, hist_b_slider])

# Middle row: kernel, show individual, and dataset (wrapped in a VBox if needed)
middle_row = HBox([kernel_dropdown, show_individual_check, dataset_radio])

# Combine everything
ui = VBox([top_row, middle_row, outlier_accordion])

# Link widgets to update function
out = interactive_output(update_kde, {
    'bandwidth': bandwidth_slider,
    'hist_b': hist_b_slider,
    'kernel_name': kernel_dropdown,
    'show_individual': show_individual_check,
    'dataset_name': dataset_radio,
    'show_outliers': show_outliers_check,
    'outlier_percentile': outlier_percentile_slider
})

# Display the UI
display(ui, out)

        
print("\n### Kernel Regression (Nadaraya‑Watson) ###")
interact(update_reg,
         bandwidth=FloatSlider(min=0.1, max=1.5, step=0.05, value=0.3,
                               description='Bandwidth h:'),
         kernel_name=Dropdown(options=list(kernels.keys()), value='Gaussian',
                              description='Kernel:'));

Output()


### Kernel Regression (Nadaraya‑Watson) ###


interactive(children=(FloatSlider(value=0.3, description='Bandwidth h:', max=1.5, min=0.1, step=0.05), Dropdow…